# LARES — ACSAC 2026 Artifact Evaluation (Google Colab)

**Paper:** *LARES: Host-centered Lateral Movement Detection via Inductive Graph Reasoning*

One linear pipeline: paste the dataset link in section 1, then **Runtime → Run all**. The
notebook clones the code and released weights, downloads the compiled dataset from Google
Drive, runs the eight evaluation commands of the README while streaming their logs, and
then compares the metrics they printed against the LARES rows of Table II.

| | |
|---|---|
| Download | 0.6 GB (`lanl_optc_datasets_compiled.tar.gz`, the compiled graph snapshots) |
| Disk | ~2 GB |
| Runtime | about an hour on a CPU runtime, less with a GPU |
| Hardware | no GPU required |

**Claim reproduced: Table II, LARES rows.** Edge-level lateral movement detection on LANL
and OpTC, in the transductive setting (Exp0) and the inductive settings where 30% and 50%
of hosts, including all malicious ones, are unseen during training (Exp1-Exp3). These rows
carry the paper's central claims: 70% precision with 104 false positives on LANL where
prior systems produce thousands, and detection that holds as hosts leave the training set.

The archive already contains the compiled graph snapshots for every experiment, so the
compilation step of section 4 is normally skipped. Pointing `DATASET_URL` at the raw
14.35 GB dataset instead also works: the notebook then compiles the snapshots itself,
which takes several hours. Restricting `DATASETS` to `['LANL']` gives the shortest run.
Everything beyond Table II (ablations, unseen-host sweeps, runtime comparisons) is
reproduced with the commands in the repository README, not in this notebook.

---
## 1. Configuration

`DATASET_URL` is a Google Drive share link to `lanl_optc_datasets_compiled.tar.gz`
(sha256 `a0b9cd2c95557061ffbc66daaad9b8a88f8820c1ac38a6127474bc57f7756981`), shared as
*Anyone with the link*. Everything else can stay as it is.

In [ ]:
import os

DATASET_URL  = ''    # e.g. 'https://drive.google.com/file/d/<id>/view?usp=sharing'

REPO_GIT_URL = 'https://github.com/TristanBilot/lares.git'
REPO_DIR     = '/content/lares'

DATASETS     = ['LANL', 'OPTC']   # ['LANL'] for the shortest complete run
EXPERIMENTS  = [0, 1, 2, 3]       # Exp0 transductive, Exp1-Exp3 inductive

DATA_ROOT = '/content/lanl_optc_datasets'
os.environ['LARES_DATA_ROOT'] = DATA_ROOT
print('Data root:', DATA_ROOT)

---
## 2. Code and dependencies

Clones the repository, whose `weights/` folder holds the released model weights, and adds
the two packages Colab does not ship. The compiled PyG extensions (`torch_scatter`,
`torch_sparse`, ...) are not needed: nothing in `src/` imports them.

In [ ]:
import subprocess

def sh(cmd, check=True):
    """Run a shell command, streaming its output into the notebook.

    subprocess writes to the kernel's file descriptors, which Colab does not show in the
    cell, so output is piped back and printed here.
    """
    print('$', cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, text=True, bufsize=1,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in proc.stdout:
        print(line, end='', flush=True)
    code = proc.wait()
    if check and code:
        raise RuntimeError(f'command failed with exit code {code}: {cmd}')
    return code

sh('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader', check=False)

if not os.path.isfile(os.path.join(REPO_DIR, 'src', 'main.py')):
    sh(f'git clone --depth 1 {REPO_GIT_URL} {REPO_DIR}')
os.chdir(REPO_DIR)

sh('pip -q install torch_geometric==2.8.0.post1 wandb==0.30.0', check=False)
os.environ['WANDB_MODE'] = 'disabled'
os.environ['WANDB_SILENT'] = 'true'

# Fail here, with a readable message, rather than inside a subprocess later.
sh('python -c "'
   'import torch, torch_geometric, sklearn, pandas, joblib, tqdm, wandb; '
   'from libauc.losses import APLoss; '
   'import sys; sys.path.insert(0, \'src\'); '
   'from models.model import Model; '
   'print(\'all imports OK\')"')

missing = [f for d in DATASETS for e in EXPERIMENTS
           if not os.path.isfile(f := f'weights/weights_{d}_inductive_exp{e}.pkl')]
assert not missing, f'weight files missing from the repository: {missing}'
print('All weights required by DATASETS/EXPERIMENTS are present.')

---
## 3. Download and extract the dataset

The archive holds the compiled snapshots of both datasets, all four experiments, with
the validation and test splits stored once and symlinked across experiments (inductive
masking only affects the training split). Skipped automatically when the data is already
unpacked, so an interrupted session can be resumed by running all cells again. If `gdown` reports a quota or permission error,
check that the file is shared as *Anyone with the link*; Google also throttles files that
were downloaded by many people in the past 24 hours.

In [ ]:
import glob

def locate_data_root():
    # A usable root holds either compiled snapshots (the archive distributed for this
    # notebook) or the raw preprocessed 1-min files (the full dataset, which section 4
    # would then compile here).
    for cand in [DATA_ROOT] + glob.glob('/content/*') + glob.glob('/content/*/*'):
        if any(os.path.isdir(os.path.join(cand, d, sub))
               for d in ['LANL', 'OPTC'] for sub in ['compiled', 'preprocessed']):
            return cand
    return None

root = locate_data_root()
if root is None:
    assert DATASET_URL, 'Set DATASET_URL in section 1 to the Google Drive link of the dataset.'
    sh('pip -q install -U gdown', check=False)
    import gdown
    os.makedirs('/content/downloads', exist_ok=True)
    archive = gdown.download(url=DATASET_URL, output='/content/downloads/', fuzzy=True)
    assert archive, ('gdown could not download the file: check the sharing setting '
                     '("Anyone with the link") and the 24h download quota.')
    print('Downloaded:', archive)
    if archive.endswith(('.tar.gz', '.tgz', '.tar')):
        sh(f'tar -xf "{archive}" -C /content')
    elif archive.endswith('.zip'):
        sh(f'unzip -q -o "{archive}" -d /content')
    else:
        raise ValueError(f'unexpected archive type: {archive}')
    root = locate_data_root()
    assert root, 'the archive did not contain LANL/preprocessed or OPTC/preprocessed'

DATA_ROOT = root
os.environ['LARES_DATA_ROOT'] = root
print('LARES_DATA_ROOT =', root)
sh(f'du -sh {root}/*', check=False)

---
## 4. Compile the graph snapshots

Normally a no-op: the distributed archive already contains the compiled snapshots for
every experiment, and anything present is skipped. This cell only does work if
`DATASET_URL` pointed at the raw 1-minute CSV dataset, in which case it runs
`src/datasets.py` once per experiment, which takes several hours.

In [ ]:
def snapshots_of(dataset, exp):
    return glob.glob(os.path.join(DATA_ROOT, dataset, 'compiled', 'test',
                                  f'{dataset}_exp{exp}', '*.pkl'))

def missing_experiments():
    return [(ds, e) for ds in DATASETS for e in EXPERIMENTS if not snapshots_of(ds, e)]

for ds, e in missing_experiments():
    sh(f'python src/datasets.py --dataset={ds} --dataset_name={ds}_exp{e} '
       f'--inductive_experiment=Exp{e}')

still = missing_experiments()
assert not still, f'compilation did not produce snapshots for {still}'
for ds in DATASETS:
    print(f'{ds}: ' + ', '.join(f'Exp{e}={len(snapshots_of(ds, e))} snapshots'
                                for e in EXPERIMENTS))

---
## 5. Run the evaluations

The eight commands of the README, section *Reproduce experiments → From weights*, one per
dataset and experiment:

```shell
python src/main.py --config=<DATASET>_inductive_exp<N> --use_weights=True
```

Each run's own log is shown below as it executes (progress bars elided). The lines to
watch are the two blocks at the end of each run: `Node detection metrics:` (stage 1,
source host detection) and `Edge detection metrics:` (stage 2, the lateral movement edges
of Table II), each ending with its `TP: x/positives | FP: y/negatives` count.

In [ ]:
import re, subprocess, time

LOG_DIR = '/content/run_logs'
os.makedirs(LOG_DIR, exist_ok=True)

def run_config(config):
    """Run one evaluation command, streaming its log, and return the output for parsing."""
    cmd = f'python src/main.py --config={config} --use_weights=True'
    print(f'$ {cmd}', flush=True)
    start = time.time()
    proc = subprocess.Popen(cmd, shell=True, cwd=REPO_DIR, text=True, bufsize=1,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    captured = []
    for raw in proc.stdout:
        line = raw.split('\r')[-1]        # tqdm rewrites its line with \r; keep the last state
        captured.append(line)
        if 'it/s' in line or 'it]' in line or '%|' in line:
            continue                       # progress bars stay out of the log
        print(line, end='', flush=True)
    code = proc.wait()
    output = ''.join(captured)
    with open(os.path.join(LOG_DIR, f'{config}.log'), 'w') as f:
        f.write(output)
    if code:
        raise RuntimeError(f'{cmd} failed with exit code {code}')
    print(f'[finished in {time.time() - start:.0f}s]\n', flush=True)
    return output

NUM = r'(-?\d+\.?\d*|nan)'

def parse_edge_metrics(stdout):
    """Extract the edge-level metrics block that each run prints (stage 2, Table II)."""
    block = stdout.split('Edge detection metrics:')[-1]
    m = re.search(rf'recall: {NUM} \| precision: {NUM} \| MCC: {NUM}', block)
    c = re.search(rf'TP: {NUM}/(\d+) \| FP: {NUM}/(\d+)', block)
    assert m and c, 'could not find the edge-level metrics in the run output'
    return {'TP': int(float(c.group(1))), 'FP': int(float(c.group(3))),
            'Recall': float(m.group(1)), 'Precision': float(m.group(2)),
            'MCC': float(m.group(3))}

print('runner ready')

In [ ]:
results = {}

for ds in DATASETS:
    for e in EXPERIMENTS:
        config = f'{ds}_inductive_exp{e}'
        print('=' * 100)
        results[(ds, f'Exp{e}')] = parse_edge_metrics(run_config(config))

print('All evaluations finished.')

---
## 5.1 Comparison with Table II (LARES rows)

Each metric printed by the runs above, side by side with the value published in Table II.
A row matches when TP and FP are identical and each ratio metric rounds to the published
two-decimal value. Any difference is listed metric by metric under the table.

In [ ]:
import pandas as pd

# Table II, LARES rows, transcribed from the paper.
PAPER_EDGE = {
    ('LANL', 'Exp0'): dict(TP=247, FP=104, Recall=0.61, Precision=0.70, MCC=0.65),
    ('LANL', 'Exp1'): dict(TP=241, FP=116, Recall=0.59, Precision=0.68, MCC=0.63),
    ('LANL', 'Exp2'): dict(TP=241, FP=116, Recall=0.59, Precision=0.68, MCC=0.63),
    ('LANL', 'Exp3'): dict(TP=241, FP=156, Recall=0.59, Precision=0.61, MCC=0.60),
    ('OPTC', 'Exp0'): dict(TP=27,  FP=63,  Recall=0.46, Precision=0.30, MCC=0.37),
    ('OPTC', 'Exp1'): dict(TP=27,  FP=63,  Recall=0.46, Precision=0.30, MCC=0.37),
    ('OPTC', 'Exp2'): dict(TP=24,  FP=66,  Recall=0.41, Precision=0.27, MCC=0.33),
    ('OPTC', 'Exp3'): dict(TP=27,  FP=63,  Recall=0.49, Precision=0.30, MCC=0.37),
}

# The paper prints the ratio metrics with two decimals, so a run value matches when it
# rounds to the published one, i.e. differs by at most half of the last printed digit.
TOL = 0.0051

rows, mismatches = [], []
for (ds, exp), run in results.items():
    paper = PAPER_EDGE[(ds, exp)]
    diffs = []
    if run['TP'] != paper['TP']:
        diffs.append(f"TP {run['TP']} vs {paper['TP']}")
    if run['FP'] != paper['FP']:
        diffs.append(f"FP {run['FP']} vs {paper['FP']}")
    for k in ['Recall', 'Precision', 'MCC']:
        if abs(run[k] - paper[k]) > TOL:
            diffs.append(f"{k} {run[k]:.3f} vs {paper[k]:.2f}")
    rows.append({
        'Dataset': ds, 'Exp': exp,
        'TP (run)': run['TP'], 'TP (paper)': paper['TP'],
        'FP (run)': run['FP'], 'FP (paper)': paper['FP'],
        'Recall (run)': round(run['Recall'], 3), 'Recall (paper)': paper['Recall'],
        'Precision (run)': round(run['Precision'], 3), 'Precision (paper)': paper['Precision'],
        'MCC (run)': round(run['MCC'], 3), 'MCC (paper)': paper['MCC'],
        'Matches paper': not diffs,
    })
    if diffs:
        mismatches.append((ds, exp, diffs))

df = pd.DataFrame(rows)
display(df)
df.to_csv('/content/lares_table2_reproduction.csv', index=False)

n = int(df['Matches paper'].sum())
print(f'{n}/{len(df)} rows match Table II (identical TP/FP, ratio metrics within '
      f'two-decimal rounding).')
for ds, exp, diffs in mismatches:
    print(f'  {ds} {exp}: ' + '; '.join(diffs))
print('Saved to /content/lares_table2_reproduction.csv')

---
## 6. Notes

This notebook reproduces the LARES rows of Table II. Everything else in the paper is run
from the repository README on the same data this notebook downloaded:

| Paper item | How to reproduce |
|---|---|
| Table VII, node level | printed by the same runs above, the `Node detection metrics:` block |
| Table III, ablations | README, *Ablation study* |
| Figures 1 and 4, unseen-host sweeps | README, *MCC @ 10-100% of unseen hosts* |
| Figure 8, hyperparameters | README, *Hyperparameter changes* |
| Figures 9 and 10, runtime and memory | requires the baselines' own repositories |

Baselines (EULER, ARGUS, JBEIL) are not rerun; their numbers come from the paper and the
repositories cited in §IV-B.

## Troubleshooting

| Symptom | Cause and fix |
|---|---|
| `gdown` fails or returns nothing | The file is not shared as *Anyone with the link*, or hit Google's 24h download quota. |
| A cell says compiled snapshots are missing | Re-run all cells; section 3 re-downloads and section 4 recompiles anything absent. |
| `CUDA out of memory` | Switch to a CPU runtime. |